# Enriquecimento dos Dados

Nesse notebook, nosso objetivo é enriquecer o nosso dataset criando duas features para `latitude` e `longitude` a partir da feature endereco.

## Importando as Bibliotecas

Vamos importar nossas bibliotecas.

In [1]:
import numpy as np
import pandas as pd

## Carregando o Dataset

Agora, vamos carregar o nosso dataset.

In [2]:
df = pd.read_parquet("../data/processed/01_cleaned.parquet")

## Enriquecendo o Dataset

Agora iremos enriquecer nosso dataset com os dados de coordenadas a partir da feature `endereco`. Primeiro vamos consultar a API do Google Maps para buscar as coordenadas dos endereços e guardar em um dicionário.

In [7]:
import os
import googlemaps

from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv()

API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")
gmaps = googlemaps.Client(key=API_KEY)

addresses = df["endereco"].unique()
address_coordinates = dict()

for address in tqdm(addresses, desc="Processing Address"):
    try:
        result = gmaps.geocode(f"{address} Goiânia, GO")

        if result:
            geometry = resultado[0]["geometry"]["location"]
            
            latitude = geometry["lat"]
            longitude = geometry["lng"]
            
            address_coordinates[address] = (latitude, longitude)
        else:
            print(f"Nenhum resultado encontrado em '{address}'")

    except Exception as e:
        print(f"Ocorreu um erro ao consultar a API: {e}")

Processing Address: 100%|██████████| 3878/3878 [21:49<00:00,  2.96it/s]


Agora vamos guardar essas informações no `pd.DataFrame`.

In [12]:
def create_coordinates_features(reg):
    coordinates = address_coordinates[reg["endereco"]]

    reg["latitude"] = coordinates[0]
    reg["longitude"] = coordinates[1]

    return reg

df = df.apply(create_coordinates_features, axis="columns")

In [17]:
df = df.drop("endereco", axis="columns")

## Convertendo os `Dtypes`

Vamos converter os `dtypes` do dataset.

In [18]:
df = df.convert_dtypes()

## Salvando o Dataset

Por fim, vamos salvar o dataset.

In [20]:
df.to_parquet("../data/processed/02_enriched.parquet")